# Wavelet-PINN for the 2D elliptic interface problem (flower / complex interface)
Mesh-free demonstration: identical solver to the circular case; only the geometry (point classification + interface normals) changes. The flux-kink magnitude now varies along the interface.

In [ ]:
from config import *
from EllipticInterface import *
from Wfamily import *
from Model import *
import math

In [ ]:
NF = len_family; n = 2*NF + 2
wi, wb, tik = 10.0, 10.0, 1e-10
def blk(rows, cM=None, cP=None, bM=None, bP=None):
    B = torch.zeros(rows, n)
    if cM is not None: B[:, :NF] = cM
    if cP is not None: B[:, NF:2*NF] = cP
    if bM is not None: B[:, 2*NF] = bM
    if bP is not None: B[:, 2*NF+1] = bP
    return B
A = torch.cat([
    blk(len(x_in),  cM=-a_in*Lin), blk(len(x_out), cP=-a_out*Lout),
    math.sqrt(wi)*blk(len(x_if), cP=Wif, cM=-Wif, bP=1., bM=-1.),
    math.sqrt(wi)*blk(len(x_if), cP=a_out*DnIf, cM=-a_in*DnIf),
    math.sqrt(wb)*blk(len(x_bc), cP=Wbc, bP=1.)], 0)
b = torch.cat([torch.full((len(x_in),), fval), torch.full((len(x_out),), fval),
    torch.zeros(len(x_if)), torch.zeros(len(x_if)), math.sqrt(wb)*torch.tensor(u_bc)]).unsqueeze(1)
s = A.norm(dim=0).clamp_min(1e-30)
theta = (torch.linalg.lstsq(torch.cat([A/s, math.sqrt(tik)*torch.eye(n)]), torch.cat([b, torch.zeros(n,1)]), driver='gelsd').solution.squeeze(1))/s
cM, cP, bM, bP = theta[:NF], theta[NF:2*NF], theta[2*NF], theta[2*NF+1]

In [ ]:
um = (Wtest@cM + bM).numpy(); up = (Wtest@cP + bP).numpy()
pred = np.where(inside_test, um, up)
relL2 = np.linalg.norm(pred - u_exact_te)/np.linalg.norm(u_exact_te)
print(f'rel L2 = {relL2:.3e} | max kink err = {np.max(np.abs(((DnIf@cP)-(DnIf@cM)).numpy() - jump_dn)):.2e}')
PR = pred.reshape(Xt.shape); UE = u_exact_te.reshape(Xt.shape); ER = np.abs(PR-UE)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for a_, Z, ttl, cm in [(ax[0],UE,'exact','viridis'),(ax[1],PR,'wavelet W-PINN (mesh-free)','viridis'),(ax[2],np.log10(ER+1e-16),'log10 |error|','magma')]:
    c = a_.contourf(Xt, Yt, Z, 40, cmap=cm); a_.plot(x_if, y_if, 'w.', ms=1); a_.set_aspect('equal'); a_.set_title(ttl); fig.colorbar(c, ax=a_)
plt.tight_layout(); plt.savefig('sol.png', dpi=120); plt.show()